Proyecto aplicado 1

Luis Felipe Rangel
Daniel Alarcón
Edward Atehortua
Angie Tamara
Frank Mendoza


# Clasificación automática de etiquetas industriales para la prevención de crosspacking

**Proyecto aplicado · Maestría en Ciencia de Datos · Universidad Icesi**
**Empresa aliada: MV Perception**

Luis Felipe Rangel · Daniel Alarcón · Edward Atehortua · Angie Tamara · Frank Mendoza

---

## El problema

En la línea de envasado se produce *crosspacking*: un producto sale con la etiqueta de una
referencia distinta a la que indica la orden de producción. La línea procesa 250 unidades por
minuto, es decir una cada 240 milisegundos, alterna 43 referencias y opera de forma continua.

Este cuaderno desarrolla y evalúa un modelo de clasificación de imágenes que identifica la
referencia a partir de su etiqueta, para contrastarla con la orden de producción.

## Las metas que fija la empresa

| Meta | Valor | Cómo se verifica en este cuaderno |
|---|---|---|
| Precisión | superior al 95 % | Sección 4, con el protocolo de partición declarado en la sección 2 |
| Tiempo de inferencia | inferior a 240 ms | Sección 5, percentil 95 sobre la cadena completa |

## Estructura del cuaderno

| Sección | Contenido |
|---|---|
| 0 | Configuración: librerías, conexión con Drive y parámetros |
| 1 | Comprensión de los datos: inventario, integridad, duplicados, procedencia y distribución |
| 2 | Preparación: depuración, protocolo de partición y preprocesamiento |
| 3 | Modelado: línea base clásica, red propia y tres arquitecturas con transferencia |
| 4 | Evaluación: métricas globales, por clase y sobre los pares difíciles |
| 5 | Latencia: medición de la cadena completa contra el presupuesto de la línea |
| 6 | Comparativo entre modelos y conclusiones |

## Cómo reproducirlo

Las imágenes no están en el repositorio: pesan unos 10 GB y viven en Google Drive.
La sección 0 monta Drive y verifica que el conjunto esté completo antes de ejecutar nada más.

Las operaciones costosas guardan su resultado en caché dentro de `pa1-analisis`, de modo que
una interrupción de Colab no obliga a repetirlas. Las particiones y las semillas quedan fijadas
en la sección 0, así que dos ejecuciones distintas producen los mismos subconjuntos y los
mismos resultados.

**Nota sobre la partición.** Las imágenes de una misma secuencia de disparos no son
observaciones independientes. Repartirlas al azar entre entrenamiento y prueba infla el
resultado: la sección 1 lo mide sobre este mismo conjunto. El protocolo de partición que usa
este cuaderno se define y se justifica en la sección 2.

## 0. Configuración

Librerías, conexión con Google Drive y parámetros del análisis. Las rutas y los umbrales
quedan reunidos en una sola celda: si cambia la ubicación del conjunto o se ajusta un criterio,
se modifica ahí y no en el resto del cuaderno.

In [1]:
import os
import re
import time
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 60)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

print('numpy', np.__version__, '· pandas', pd.__version__, '· Pillow', Image.__version__)

numpy 2.1.3 · pandas 2.2.3 · Pillow 11.3.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ---------- Rutas. Ajustar si las carpetas tienen otro nombre en Drive ----------
RAIZ   = Path('/content/drive/MyDrive/pa1-imagenes')    # contiene las 43 carpetas de referencias
SALIDA = Path('/content/drive/MyDrive/pa1-analisis')    # caché, tablas y figuras
SALIDA.mkdir(parents=True, exist_ok=True)

# ---------- Reproducibilidad ----------
SEMILLA = 42
np.random.seed(SEMILLA)
rng = np.random.default_rng(SEMILLA)

# ---------- Umbrales del análisis exploratorio ----------
CORTE_SESION = 600     # segundos sin disparos que definen otra jornada de trabajo
CORTE_RAFAGA = 3       # segundos sin disparos que definen otra ráfaga
RMSE_CASI    = 4.0     # error cuadrático medio bajo el cual dos imágenes son casi la misma
LADO_MINI    = 256     # lado largo de la miniatura usada para comparar imágenes

# ---------- Parámetros de la partición ----------
FRAC_PRUEBA = 0.30     # proporción de ráfagas que van a prueba en las clases de una sola jornada
FRAC_VALID  = 0.20     # proporción de ráfagas de entrenamiento que van a validación

# ---------- Características del montaje, usadas para interpretar resoluciones ----------
ANCHO_NATIVO = 1440
ALTO_NATIVO  = 944
MM_POR_PIXEL = 240 / ANCHO_NATIVO   # campo de visión horizontal de 240 mm

print(f'origen : {RAIZ}')
print(f'salida : {SALIDA}')
print(f'escala : {MM_POR_PIXEL:.4f} mm por píxel')

origen : /content/drive/MyDrive/pa1-imagenes
salida : /content/drive/MyDrive/pa1-analisis
escala : 0.1667 mm por píxel


In [7]:
assert RAIZ.exists(), f'No existe la ruta {RAIZ}. Revisa el nombre de la carpeta en Drive.'

carpetas = sorted(
    [d for d in RAIZ.iterdir() if d.is_dir() and not d.name.startswith(('_', '.'))],
    key=lambda d: int(d.name.split('_')[0]) if d.name.split('_')[0].isdigit() else 999
)

conteo = {c.name: len([f for f in os.listdir(c) if f.lower().endswith('.bmp')]) for c in carpetas}
total = sum(conteo.values())

print(f'{len(carpetas)} carpetas · {total} imágenes BMP\n')
for nombre, n in conteo.items():
    print(f'  {nombre:36s} {n:>5}')

ESPERADAS_CARPETAS = 43
ESPERADAS_IMAGENES = 2607   # 2.605 del conjunto original más dos archivos que añadió la subida
if len(carpetas) != ESPERADAS_CARPETAS or total != ESPERADAS_IMAGENES:
    print(f'\nAVISO: se esperaban {ESPERADAS_CARPETAS} carpetas y {ESPERADAS_IMAGENES} imágenes.')
    print('Si la subida a Drive todavía no termina, conviene esperar antes de continuar.')
else:
    print('\nEl conjunto está completo.')

43 carpetas · 2607 imágenes BMP

  0_MULTIVIAMINAS_SAVITAL_510             37
  1_ANTICASPA_SAVITAL_510                 74
  2_COLAGENO_SAVITAL_510                  64
  3_FUSION_SAVITAL_490                    41
  4_FUSION_SAVITAL_510                   107
  5_HIALURONICO_SAVITAL_510               31
  6_MULTIOLEOS_SAVITAL_510                24
  7_SERUM_SAVITAL_490                     51
  8_ARGAN_SAVITAL_510                     41
  9_ANTICASPA_LIMPIEZA_SAVITAL_510       133
  10_COLAGENO_SAVITAL_510                 35
  11_SERUM_SAVITAL_490                    47
  12_ELIXIR_SAVITAL_510                   19
  13_DOVE_NUTRICION_370                  135
  14_DOVE_RECONSTRUCCION_370              49
  15_DOVE_REGENERACION_370                92
  16_DOVE_REPAIR_350                      87
  17_DOVE_HIDRA_SUAVE_370                111
  18_DOVE_HIDRA_370                       24
  19_SEDAL_UV_340                         52
  20_SEDAL_ZERO_CASPA_340                 95
  21_SEDAL_RIZOS_DEFIN

## 1. Comprensión de los datos

Esta sección reconstruye el inventario del conjunto desde los archivos, verifica su
integridad, identifica duplicados, reconstruye la procedencia temporal de cada imagen y
mide el desbalance entre clases.

Cierra con la medida que más condiciona el resto del cuaderno: cuánto cambia el resultado
según cómo se reparta el conjunto entre entrenamiento y prueba.

### 1.1 Inventario

Una sola pasada sobre las 2.607 imágenes. En ese recorrido se calcula, por archivo:

- el **resumen MD5** del contenido, que identifica archivos idénticos byte a byte;
- la **cabecera del BMP**, que da resolución, modo de color y número de canales;
- la **marca de tiempo** codificada en el nombre, que da la procedencia temporal;
- las **estadísticas fotométricas** por canal, que miden la variación de iluminación;
- una **miniatura en gris**, que permite comparar imágenes entre sí sin volver a leer los 10 GB.

Hacerlo todo en un solo recorrido evita leer el conjunto cinco veces. El resultado queda en
caché, de modo que una desconexión de Colab no obliga a repetir la pasada.

In [8]:
PATRON = re.compile(r'^Imagen?_(\d+)_(\d{14})', re.IGNORECASE)
W_MINI = LADO_MINI
H_MINI = int(round(LADO_MINI * ALTO_NATIVO / ANCHO_NATIVO))

csv_inv = SALIDA / 'inventario.csv'
npz_min = SALIDA / 'miniaturas.npz'

def md5_archivo(ruta, bloque=1 << 20):
    h = hashlib.md5()
    with open(ruta, 'rb') as f:
        for trozo in iter(lambda: f.read(bloque), b''):
            h.update(trozo)
    return h.hexdigest()

if csv_inv.exists() and npz_min.exists():
    inv = pd.read_csv(csv_inv, parse_dates=['fecha'])
    z = np.load(npz_min, allow_pickle=True)
    MINI, CLAVES = z['x'], z['claves']
    print(f'caché leída: {len(inv)} imágenes')
else:
    filas, minis, claves = [], [], []
    t0 = time.time()
    for k, carpeta in enumerate(carpetas, 1):
        archivos = sorted([f for f in os.listdir(carpeta) if f.lower().endswith('.bmp')])
        for nombre in archivos:
            ruta = carpeta / nombre
            m = PATRON.match(nombre)
            im = Image.open(ruta); im.load()
            ancho, alto, modo, canales = im.size[0], im.size[1], im.mode, len(im.getbands())
            if ancho < alto:                                  # corrige la imagen girada
                im = im.transpose(Image.ROTATE_270)
            rgb  = np.asarray(im.convert('RGB'), dtype=np.float32)
            gris = rgb.mean(axis=2)
            minis.append(np.asarray(
                Image.fromarray(gris.astype('uint8')).resize((W_MINI, H_MINI), Image.LANCZOS),
                dtype=np.uint8))
            claves.append(f'{carpeta.name}/{nombre}')
            filas.append({
                'carpeta': carpeta.name, 'archivo': nombre, 'bytes': ruta.stat().st_size,
                'disparo': int(m.group(1)) if m else np.nan,
                'fecha': pd.to_datetime(m.group(2), format='%Y%m%d%H%M%S') if m else pd.NaT,
                'ancho': ancho, 'alto': alto, 'modo': modo, 'canales': canales,
                'md5': md5_archivo(ruta),
                'media_r': rgb[:, :, 0].mean(), 'media_g': rgb[:, :, 1].mean(),
                'media_b': rgb[:, :, 2].mean(), 'std_gris': gris.std(),
                'frac_saturado': float((gris >= 254).mean()),
            })
        mins = (time.time() - t0) / 60
        print(f'[{k:>2}/{len(carpetas)}] {carpeta.name:36s} {len(archivos):>4}  '
              f'· {len(filas):>5} acumuladas · {mins:.1f} min', flush=True)

    inv = pd.DataFrame(filas)
    MINI, CLAVES = np.stack(minis), np.array(claves)
    inv.to_csv(csv_inv, index=False)
    np.savez_compressed(npz_min, x=MINI, claves=CLAVES)
    print(f'\ncaché escrita en {SALIDA} · {(time.time() - t0) / 60:.1f} min')

MINI = MINI.reshape(len(MINI), -1).astype(np.float32)
print(f'inventario: {inv.shape[0]} filas × {inv.shape[1]} columnas')
print(f'miniaturas: {MINI.shape[0]} × {W_MINI}×{H_MINI}')

[ 1/43] 0_MULTIVIAMINAS_SAVITAL_510            37  ·    37 acumuladas · 0.5 min
[ 2/43] 1_ANTICASPA_SAVITAL_510                74  ·   111 acumuladas · 1.3 min
[ 3/43] 2_COLAGENO_SAVITAL_510                 64  ·   175 acumuladas · 2.2 min
[ 4/43] 3_FUSION_SAVITAL_490                   41  ·   216 acumuladas · 2.9 min
[ 5/43] 4_FUSION_SAVITAL_510                  107  ·   323 acumuladas · 3.4 min
[ 6/43] 5_HIALURONICO_SAVITAL_510              31  ·   354 acumuladas · 3.8 min


KeyboardInterrupt: 